# Go2 Course Homework: Teaching Colab Template

Welcome to Assignment 1.

It assumes that the readable homework package is stored in a course GitHub repo: https://github.com/WeijieLai1024/EEC289A_Robotics-Homework.git
This version keeps the important
source files visible as normal `.py` / `.json` files.

This release provides baseline intentionally only
learns `{stand, +vx}`. Students then extend the same pipeline so the robot can
track forward / backward motion, lateral motion, and yaw commands.

`stage_1` and `stage_2` are internal training curriculum stages, not two
separate homework assignments.

Recommended lecture / demo usage:
1. run setup
2. inspect the environment
3. show the key files
4. train or restore a checkpoint
5. run the public benchmark and discuss the axis-wise metrics


## 1. Configure repository URLs

Set `COURSE_REPO_URL` to the GitHub repository that contains this readable
homework package. The repo should contain:
- `train.py`
- `test_policy.py`
- `generate_public_rollout.py`
- `public_eval.py`
- `go2_pg_env/`
- `configs/course_config.json`

Important Notice: Since this assignment runs in Google Colab, you should not treat the Colab runtime as permanent storage. Before you start, create your own GitHub repository by either forking this repo or creating a new repo based on it, then change the notebook’s course_repo_url to point to your own repository instead of the course repo. From that point on, all of your work should be done in the copy cloned from your own repository. After making any changes in Colab, you should regularly save them back to GitHub using git so that your code is stored safely and remains reproducible. In short: replace the repo URL with your own, do all development in that cloned copy, and push your changes to GitHub regularly rather than relying on Colab to keep your files.

In [2]:
from pathlib import Path
import io
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
from urllib.parse import urlparse

COURSE_REPO_URL = "https://github.com/aooooofu-code/EEC289A_Robotics-Homework.git"
COURSE_REPO_BRANCH = "main"
COURSE_REPO_DIR = Path("/content/go2_course_repo")

PLAYGROUND_REPO = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF = "dd38c285c6d54266287081e516109f0b15985818"

UNITREE_MUJOCO_REPO = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF = "1a37b051a10be723405b7ed6dc839361af036d88"

PLAYGROUND_DIR = Path("/content/mujoco_playground")
UNITREE_DIR = Path("/content/unitree_mujoco")

MENAGERIE_REPO = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF = "1b86ece576591213e2b666ebf59508454200ca97"
MENAGERIE_DIR = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"

def run(cmd):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)

def github_archive_url(repo_url: str, ref: str) -> str:
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"

def download_repo_snapshot(repo_url: str, ref: str, target_dir: Path) -> None:
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [path for path in tmp_dir.iterdir() if path.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted directory, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def checkout_existing_repo(target_dir: Path, ref: str) -> None:
    try:
        run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git fetch failed for {target_dir}: {exc}. Trying local checkout.")
    run(["git", "-C", target_dir, "checkout", ref])

def ensure_pinned_repo(repo_url: str, ref: str, target_dir: Path) -> None:
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            checkout_existing_repo(target_dir, ref)
            return
        except subprocess.CalledProcessError as exc:
            print(f"[warn] local git checkout failed for {target_dir}: {exc}. Re-downloading snapshot.")
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)

    try:
        run(["git", "clone", repo_url, target_dir])
        checkout_existing_repo(target_dir, ref)
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git path failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)

def ensure_course_repo(repo_url: str, branch: str, target_dir: Path) -> None:
    if target_dir.exists():
        return
    try:
        run(["git", "clone", repo_url, target_dir])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git clone failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, branch, target_dir)

if "google.colab" in sys.modules:
    print("Running inside Colab.")
else:
    print("This notebook was designed for Colab, but local execution may also work.")


Running inside Colab.


## 2. Install system packages and clone repositories

In [3]:
!command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y ffmpeg)
!python -m pip install -q -U pip setuptools wheel "jedi>=0.16"

import importlib.util
if importlib.util.find_spec("playground") is not None:
    !python -m pip uninstall -y playground

ensure_pinned_repo(PLAYGROUND_REPO, PLAYGROUND_REF, PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)
ensure_pinned_repo(MENAGERIE_REPO, MENAGERIE_REF, MENAGERIE_DIR)

!python -m pip install -q -r {COURSE_REPO_DIR / "configs" / "colab_requirements.txt"}

%cd {PLAYGROUND_DIR}
!python -m pip install -q -e .

%cd {COURSE_REPO_DIR}
print("Setup finished. Some Colab dependency warnings can usually be ignored if the later checks pass.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 60.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.
+ git clone https://github.com/google-deepmind/mujoco_playground.git /content/mujoco_playground
+ git -C /content/mujoco_playground fetch --all --tags
+ git -C /content/mujoco_playground checkout dd38c285c6d54266287081e516109f0b15985818
+ git clone https://github.com/unitreerobotics/unitree_mujoco.git /content/unitree_mujoco
+ git -C /content/unitree_mujoco fetch --all --tags
+ git -C /content/unitree_mujoco checkout 1a37b051a10be723405b7ed6dc839361af036d88
+ git clone https://github.com/aoooo

## 3. Copy Go2 assets from `unitree_mujoco` into the local course environment

Use the repo-side helper script so the notebook stays aligned with the code
students will download and edit.


In [4]:
%cd {COURSE_REPO_DIR}
!python scripts/copy_go2_assets.py --unitree-dir {UNITREE_DIR} --course-dir {COURSE_REPO_DIR}


/content/go2_course_repo
Copied 16 assets into /content/go2_course_repo/go2_pg_env/xmls/assets


## 4. Inspect the environment before training

Run this before creating the Colab runtime override file. The default course
config already contains the released forward-only baseline setup.


In [5]:
%cd {COURSE_REPO_DIR}
!python inspect_env.py --stage-name stage_1


/content/go2_course_repo
{
  "environment_name": "Go2JoystickFlatTerrain",
  "stage_name": "stage_1",
  "backend_impl": "jax",
  "control_dt": 0.02,
  "sim_dt": 0.004,
  "episode_length": 1000,
  "action_size": 12,
  "actor_obs_size": 48,
  "critic_obs_size": 123,
  "observation_layout": {
    "state": [
      [
        "local_linvel",
        3
      ],
      [
        "gyro",
        3
      ],
      [
        "gravity",
        3
      ],
      [
        "joint_pos_error",
        12
      ],
      [
        "joint_vel",
        12
      ],
      [
        "last_action",
        12
      ],
      [
        "command",
        3
      ]
    ],
    "privileged_state_extra": [
      [
        "gyro_clean",
        3
      ],
      [
        "accelerometer",
        3
      ],
      [
        "gravity_clean",
        3
      ],
      [
        "local_linvel_clean",
        3
      ],
      [
        "global_angvel",
        3
      ],
      [
        "joint_pos_error_clean",
        12
 

## 5. Read the most important files

Students should focus on:
1. `go2_pg_env/joystick.py`
2. `configs/course_config.json`
3. `benchmark_specs.py`
4. `public_eval.py`
5. `train.py`


In [6]:
!sed -n '1,260p' go2_pg_env/joystick.py


"""Joystick locomotion task for the local Go2 environment.

This task is adapted from MuJoCo Playground's Go1 joystick task. The local
changes are intentionally small so that students can compare the official
baseline against a course-specific Go2 variant.

Observation summary
-------------------
state (actor input):
    [local_linvel(3), gyro(3), gravity(3),
     joint_pos_error(12), joint_vel(12),
     last_action(12), command(3)]  -> 48 dims

privileged_state (critic-only input during training):
    state + extra simulator-only signals -> 123 dims

Action summary
--------------
The policy outputs 12 joint offsets. The final motor target is:
    target_joint_pos = default_pose + action_scale * policy_action
"""

from __future__ import annotations

from typing import Any, Dict, Optional, Union

import jax
import jax.numpy as jp
from ml_collections import config_dict
from mujoco import mjx
from mujoco.mjx._src import math
import numpy as np

from mujoco_playground._src import mjx_env



In [7]:
%cd /content/go2_course_repo
!git checkout -- go2_pg_env/joystick.py

/content/go2_course_repo


In [8]:
!sed -n '240,255p' /content/go2_course_repo/go2_pg_env/joystick.py


        rng, key1, key2 = jax.random.split(rng, 3)
        time_until_next_cmd = jax.random.exponential(key1) * 5.0
        steps_until_next_cmd = jp.round(time_until_next_cmd / self.dt).astype(jp.int32)
        command = jax.random.uniform(key2, shape=(3,), minval=self._cmd_min, maxval=self._cmd_max)

        info = {
            "rng": rng,
            "command": command,
            "steps_until_next_cmd": steps_until_next_cmd,
            "last_act": jp.zeros(self.mjx_model.nu),
            "last_last_act": jp.zeros(self.mjx_model.nu),
            "feet_air_time": jp.zeros(4),
            "last_contact": jp.zeros(4, dtype=bool),
            "swing_peak": jp.zeros(4),
            "steps_until_next_pert": steps_until_next_pert,


In [10]:
from pathlib import Path

path = Path("/content/go2_course_repo/go2_pg_env/joystick.py")
text = path.read_text()

old = "        command = jax.random.uniform(key2, shape=(3,), minval=self._cmd_min, maxval=self._cmd_max)\n"

new = """        if self._command_stage_name == "stage_2":
            cmd_min = self._student_stage2_goal_min
            cmd_max = self._student_stage2_goal_max
        else:
            cmd_min = self._cmd_min
            cmd_max = self._cmd_max

        command = jax.random.uniform(
            key2, shape=(3,), minval=cmd_min, maxval=cmd_max
        )
"""

assert old in text, "Original command line not found. Restore joystick.py first."
path.write_text(text.replace(old, new))
print("Updated joystick.py for ours.")

Updated joystick.py for ours.


In [ ]:
!sed -n '1,220p' train.py
!printf '\n===== configs/course_config.json =====\n'
!sed -n '1,240p' configs/course_config.json


In [ ]:
!sed -n '1,220p' benchmark_specs.py
!printf '\n===== public_eval.py =====\n'
!sed -n '1,240p' public_eval.py


#!/usr/bin/env python3
"""Deterministic command scripts used for demo videos and public benchmark runs."""

from __future__ import annotations

from typing import Any

import numpy as np


PUBLIC_EPISODE_LABELS = (
    "forward_only",
    "lateral_only",
    "yaw_only",
    "combined",
)


def build_demo_segments(config: dict[str, Any]) -> list[list[float]]:
    """Return an easy-to-read command script for demo videos.

    The demo is intentionally human-readable:
    stand -> slow forward -> medium forward -> fast forward -> slow forward -> stand.
    """
    demo_cfg = config["demo_rollout"]
    if "segments" in demo_cfg and demo_cfg["segments"]:
        return [[float(x) for x in segment] for segment in demo_cfg["segments"]]
    return [
        [0.0, 0.0, 0.0],
        [0.20, 0.0, 0.0],
        [0.40, 0.0, 0.0],
        [0.60, 0.0, 0.0],
        [0.30, 0.0, 0.0],
        [0.0, 0.0, 0.0],
    ]


def public_command_script(safe_ranges: dict[str, list[float]], episode_idx: int) -> li

## 6. Define a Colab-friendly runtime config

Write a small `runtime_overrides` block on top of the base course config.
This keeps the released homework settings intact while still letting Colab
use a smaller or larger training profile.


In [11]:
import json

runtime_overrides = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [256, 256, 128],
    "value_hidden_layer_sizes": [256, 256, 128],
    "stage_1_num_timesteps": 10_000_000,
    "stage_2_num_timesteps": 5_000_000,
}

config_path = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
base_config_path = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config = json.loads(base_config_path.read_text())
base_config["runtime_overrides"] = runtime_overrides
config_path.write_text(json.dumps(base_config, indent=2))
print("wrote", config_path)


wrote /content/go2_course_repo/configs/colab_runtime_config.json


## 7. Dry-run the training config

In [12]:
!python train.py --config configs/colab_runtime_config.json --dry-run

{
  "homework_name": "Homework: Sim-to-Real Quadruped Locomotion with MuJoCo Playground",
  "robot": "Go2",
  "environment_name": "Go2JoystickFlatTerrain",
  "framework": "MuJoCo Playground + Brax PPO + MJX",
  "backend_impl": "jax",
  "actor_obs_key": "state",
  "critic_obs_key": "privileged_state",
  "use_domain_randomization": true,
  "seed": 0,
  "control": {
    "ctrl_dt": 0.02,
    "sim_dt": 0.004,
    "action_scale": 0.5,
    "action_type": "absolute_joint_position_target",
    "torque_mapping": "position_target_through_pd_actuator"
  },
  "course_budget": {
    "baseline_total_env_steps": 15000000,
    "leaderboard_max_env_steps": 30000000,
    "flat_terrain_only": true,
    "require_colab_gpu_runtime": true
  },
  "training_defaults": {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [
      256,
      256,
      128
    ],
    "value_hidden_layer_sizes": [
      256,
      256,
      128
    ]
  },
  

## 8. Run training

This command trains the **released forward-only baseline**.
Students should later modify `go2_pg_env/joystick.py` and possibly
`configs/course_config.json`, then rerun training and compare benchmark
metrics.


In [13]:
%cd {COURSE_REPO_DIR}

RUN_NAME = "run_baseline"
OUTPUT_DIR = COURSE_REPO_DIR / "artifacts" / RUN_NAME

%cd {COURSE_REPO_DIR}

!python train.py \
  --config configs/colab_runtime_config.json \
  --stage both \
  --output-dir {OUTPUT_DIR}


/content/go2_course_repo
/content/go2_course_repo
[run] output_dir=/content/go2_course_repo/artifacts/run_baseline
[run] stages=['stage_1', 'stage_2']
[stage_1] starting train: env=Go2JoystickFlatTerrain impl=jax target_steps=10000000 num_envs=1024 batch_size=256 num_evals=5
[stage_1] steps=0 eval_reward=0.005
[stage_1] steps=3276800 eval_reward=1.463
[stage_1] steps=6553600 eval_reward=21.036
[stage_1] steps=9830400 eval_reward=26.875
[stage_1] steps=13107200 eval_reward=29.463
[stage_1] finished: latest_checkpoint=/content/go2_course_repo/artifacts/run_baseline/stage_1/checkpoints/000013107200 selected_checkpoint_source=/content/go2_course_repo/artifacts/run_baseline/stage_1/checkpoints/000013107200
[stage_2] starting train: env=Go2JoystickFlatTerrain impl=jax target_steps=5000000 num_envs=1024 batch_size=256 num_evals=5
[stage_2] restoring from checkpoint: /content/go2_course_repo/artifacts/run_baseline/stage_1/checkpoints/000013107200
[stage_2] steps=0 eval_reward=0.379
[stage_2] s

In [17]:
RUN_NAME = "run_ours"
OUTPUT_DIR = COURSE_REPO_DIR / "artifacts" / RUN_NAME

%cd {COURSE_REPO_DIR}

!python train.py \
  --config configs/colab_runtime_config.json \
  --stage both \
  --output-dir {OUTPUT_DIR}

/content/go2_course_repo
[run] output_dir=/content/go2_course_repo/artifacts/run_ours
[run] stages=['stage_1', 'stage_2']
[stage_1] starting train: env=Go2JoystickFlatTerrain impl=jax target_steps=10000000 num_envs=1024 batch_size=256 num_evals=5
[stage_1] steps=0 eval_reward=0.005
[stage_1] steps=3276800 eval_reward=2.012
[stage_1] steps=6553600 eval_reward=21.856
[stage_1] steps=9830400 eval_reward=26.546
[stage_1] steps=13107200 eval_reward=28.873
[stage_1] finished: latest_checkpoint=/content/go2_course_repo/artifacts/run_ours/stage_1/checkpoints/000013107200 selected_checkpoint_source=/content/go2_course_repo/artifacts/run_ours/stage_1/checkpoints/000013107200
[stage_2] starting train: env=Go2JoystickFlatTerrain impl=jax target_steps=5000000 num_envs=1024 batch_size=256 num_evals=5
[stage_2] restoring from checkpoint: /content/go2_course_repo/artifacts/run_ours/stage_1/checkpoints/000013107200
[stage_2] steps=0 eval_reward=0.412
[stage_2] steps=1638400 eval_reward=11.159
[stage_2]

## 9. Restore a checkpoint and render a deterministic demo

The demo script still contains turn / strafe / combined segments on purpose.
A released forward-only baseline is expected to look okay on standing / forward
segments and much worse on the others until students extend the command sampler.


In [21]:
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "run_ours" / "best_checkpoint"
DEMO_DIR = COURSE_REPO_DIR / "artifacts" / "demo_bundle"

%cd {COURSE_REPO_DIR}

!python test_policy.py \
  --config configs/colab_runtime_config.json \
  --checkpoint-dir {CHECKPOINT_DIR} \
  --stage-name stage_2 \
  --output-dir {DEMO_DIR}


/content/go2_course_repo
100% 751/751 [07:55<00:00,  1.58it/s]
{
  "homework_name": "Homework: Sim-to-Real Quadruped Locomotion with MuJoCo Playground",
  "robot": "Go2",
  "environment_name": "Go2JoystickFlatTerrain",
  "stage_name": "stage_2",
  "checkpoint_dir": "/content/go2_course_repo/artifacts/run_ours/best_checkpoint",
  "num_steps": 750,
  "metrics": {
    "velocity_tracking_error": 0.09355297684669495,
    "yaw_tracking_error": 0.06962867081165314,
    "fall_rate": 0.0,
    "energy_proxy": 19.562156677246094,
    "foot_slip_proxy": 0.196212038397789
  },
  "normalized_scores": {
    "velocity_tracking_error": 1.0,
    "yaw_tracking_error": 1.0,
    "fall_rate": 1.0,
    "energy_proxy": 0.6386826038360596,
    "foot_slip_proxy": 0.021044231123394496
  },
  "course_composite_score": 0.8479068136877485,
  "rollout_summary": {
    "video_path": "/content/go2_course_repo/artifacts/demo_bundle/demo.mp4",
    "rollout_npz": "/content/go2_course_repo/artifacts/demo_bundle/rollout_pub

In [24]:
from IPython.display import Video, display

video_path = "/content/go2_course_repo/artifacts/demo_bundle/demo.mp4"
display(Video(video_path, embed=True, width=640))

## 10. Generate the public benchmark rollout

The public benchmark contains four deterministic cases:
1. forward / backward `vx`
2. lateral `vy`
3. yaw
4. combined `vx + vy + yaw`

`public_eval.py` reports axis-wise errors so students can analyze exactly
which capabilities improved and which did not.


In [ ]:
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "run_baseline" / "best_checkpoint"
PUBLIC_DIR = COURSE_REPO_DIR / "artifacts" / "public_eval_bundle_baseline"

%cd {COURSE_REPO_DIR}

!python generate_public_rollout.py \
  --config configs/colab_runtime_config.json \
  --checkpoint-dir {CHECKPOINT_DIR} \
  --stage-name stage_2 \
  --output-dir {PUBLIC_DIR} \
  --num-episodes 4 \
  --render-first-episode

!python public_eval.py \
  --config configs/colab_runtime_config.json \
  --rollout-npz {PUBLIC_DIR / "rollout_public_eval.npz"} \
  --output-json {PUBLIC_DIR / "public_eval.json"}

/content/go2_course_repo
100% 1501/1501 [16:37<00:00,  1.51it/s]
{
  "checkpoint_dir": "/content/go2_course_repo/artifacts/run_baseline/best_checkpoint",
  "stage_name": "stage_2",
  "num_episodes": 4,
  "episode_length_steps": 1500,
  "episode_lengths_realized": [
    1500,
    401,
    404,
    400
  ],
  "episode_labels": [
    "forward_only",
    "lateral_only",
    "yaw_only",
    "combined"
  ],
  "rollout_npz": "/content/go2_course_repo/artifacts/public_eval_bundle_baseline/rollout_public_eval.npz",
  "video_path": "/content/go2_course_repo/artifacts/public_eval_bundle_baseline/public_eval_episode0.mp4"
}
{
  "homework_name": "Homework: Sim-to-Real Quadruped Locomotion with MuJoCo Playground",
  "robot": "Go2",
  "environment_name": "Go2JoystickFlatTerrain",
  "num_steps": 2705,
  "num_episodes": 4,
  "metrics": {
    "velocity_tracking_error": 0.08481015264987946,
    "yaw_tracking_error": 0.0971708819270134,
    "fall_rate": 0.75,
    "energy_proxy": 12.27889633178711,
    "fo

In [23]:
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "run_ours" / "best_checkpoint"
PUBLIC_DIR = COURSE_REPO_DIR / "artifacts" / "public_eval_bundle_ours"

%cd {COURSE_REPO_DIR}

!python generate_public_rollout.py \
  --config configs/colab_runtime_config.json \
  --checkpoint-dir {CHECKPOINT_DIR} \
  --stage-name stage_2 \
  --output-dir {PUBLIC_DIR} \
  --num-episodes 4 \
  --render-first-episode

!python public_eval.py \
  --config configs/colab_runtime_config.json \
  --rollout-npz {PUBLIC_DIR / "rollout_public_eval.npz"} \
  --output-json {PUBLIC_DIR / "public_eval.json"}

/content/go2_course_repo
100% 1501/1501 [15:36<00:00,  1.60it/s]
{
  "checkpoint_dir": "/content/go2_course_repo/artifacts/run_ours/best_checkpoint",
  "stage_name": "stage_2",
  "num_episodes": 4,
  "episode_length_steps": 1500,
  "episode_lengths_realized": [
    1500,
    1500,
    1500,
    1500
  ],
  "episode_labels": [
    "forward_only",
    "lateral_only",
    "yaw_only",
    "combined"
  ],
  "rollout_npz": "/content/go2_course_repo/artifacts/public_eval_bundle_ours/rollout_public_eval.npz",
  "video_path": "/content/go2_course_repo/artifacts/public_eval_bundle_ours/public_eval_episode0.mp4"
}
{
  "homework_name": "Homework: Sim-to-Real Quadruped Locomotion with MuJoCo Playground",
  "robot": "Go2",
  "environment_name": "Go2JoystickFlatTerrain",
  "num_steps": 6000,
  "num_episodes": 4,
  "metrics": {
    "velocity_tracking_error": 0.06358716636896133,
    "yaw_tracking_error": 0.09334202110767365,
    "fall_rate": 0.0,
    "energy_proxy": 7.18351411819458,
    "foot_slip_pr

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


baseline_path = Path("/content/go2_course_repo/artifacts/public_eval_bundle_baseline/public_eval.json")
ours_path = Path("/content/go2_course_repo/artifacts/public_eval_bundle_ours/public_eval.json")


with open(baseline_path) as f:
    baseline = json.load(f)

with open(ours_path) as f:
    ours = json.load(f)

metrics = [
    "velocity_tracking_error",
    "yaw_tracking_error",
    "fall_rate",
    "energy_proxy",
    "foot_slip_proxy",
]

plt.figure(figsize=(8,5))
x = np.arange(len(metrics))
width = 0.35

baseline_vals = [baseline["metrics"][m] for m in metrics]
ours_vals = [ours["metrics"][m] for m in metrics]

plt.bar(x - width/2, baseline_vals, width, label="baseline")
plt.bar(x + width/2, ours_vals, width, label="ours")

plt.xticks(x, metrics, rotation=25)
plt.ylabel("Value")
plt.title("Overall Metrics Comparison")
plt.legend()
plt.tight_layout()
plt.savefig("/content/overall_metrics.png", dpi=200)
plt.show()


baseline_eps = pd.DataFrame(baseline["per_episode_summary"])
ours_eps = pd.DataFrame(ours["per_episode_summary"])

labels = baseline_eps["episode_label"].tolist()
x = np.arange(len(labels))

for metric in ["velocity_tracking_error", "yaw_tracking_error"]:
    plt.figure(figsize=(8,5))
    plt.bar(x - width/2, baseline_eps[metric], width, label="baseline")
    plt.bar(x + width/2, ours_eps[metric], width, label="ours")

    plt.xticks(x, labels)
    plt.ylabel(metric)
    plt.title(f"Per-direction {metric}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"/content/per_direction_{metric}.png", dpi=200)
    plt.show()


plt.figure(figsize=(8,5))
plt.bar(x - width/2, baseline_eps["fell"].astype(int), width, label="baseline")
plt.bar(x + width/2, ours_eps["fell"].astype(int), width, label="ours")

plt.xticks(x, labels)
plt.ylabel("Fall (1=yes, 0=no)")
plt.title("Per-direction Fall Behavior")
plt.legend()
plt.tight_layout()
plt.savefig("/content/fall_behavior.png", dpi=200)
plt.show()


for metric in ["energy_proxy", "foot_slip_proxy"]:
    plt.figure(figsize=(6,4))
    plt.bar(["baseline", "ours"], [baseline["metrics"][metric], ours["metrics"][metric]])
    plt.title(metric)
    plt.ylabel(metric)
    plt.tight_layout()
    plt.savefig(f"/content/{metric}.png", dpi=200)
    plt.show()

print("All plots saved to /content/")

In [ ]:
%cd /content/go2_course_repo

git status

In [20]:
!du -sh artifacts/run_ours/best_checkpoint

968K	artifacts/run_ours/best_checkpoint


In [25]:
%cd /content/go2_course_repo

!git status

!git add .
!git commit -m "Add HW1 multi-direction locomotion results"
!git push origin main

/content/go2_course_repo
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   go2_pg_env/joystick.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	configs/colab_runtime_config.json

no changes added to commit (use "git add" and/or "git commit -a")
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@0ba5ed88ce9c.(none)')
fatal: could not read Username for 'https://github.com': No such device or address


In [26]:
!cp "/content/Assignment_1_Colab (1).ipynb" \
"/content/go2_course_repo/notebooks/Assignment_1_Colab_Ao.ipynb"

cp: cannot stat '/content/Assignment_1_Colab (1).ipynb': No such file or directory


In [27]:
!ls /content/go2_course_repo/notebooks

go2_public_colab_template.ipynb


In [28]:
!find /content -name "*.ipynb"

/content/go2_course_repo/notebooks/go2_public_colab_template.ipynb
/content/mujoco_playground/learning/notebooks/vision.ipynb
/content/mujoco_playground/learning/notebooks/locomotion.ipynb
/content/mujoco_playground/learning/notebooks/manipulation.ipynb
/content/mujoco_playground/learning/notebooks/dm_control_suite.ipynb
/content/mujoco_playground/mujoco_playground/external_deps/mujoco_menagerie/tutorial.ipynb
/content/mujoco_playground/mujoco_playground/experimental/learning/viz_sim_policies.ipynb
/content/mujoco_playground/mujoco_playground/experimental/learning/getup.ipynb
/content/mujoco_playground/mujoco_playground/experimental/learning/phase.ipynb
/content/mujoco_playground/mujoco_playground/experimental/learning/apollo_joystick.ipynb
/content/mujoco_playground/mujoco_playground/experimental/learning/open_cabinet.ipynb
/content/mujoco_playground/mujoco_playground/experimental/learning/position_cube.ipynb
/content/mujoco_playground/mujoco_playground/experimental/learning/cube_rota

In [ ]:
%cd /content/go2_course_repo

In [ ]:
!git add .
!git commit -m "HW1 multi-direction locomotion with checkpoint and notebook"
!git push origin main

In [ ]:
!git remote -v